In [ ]:
import time from pathlib
import Path 
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.preprocessing import StandardScaler 
from sklearn.neural_network import MLPClassifier 
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay 
RANDOM_STATE = 42 
DATA_DIR = Path("data") / "UCI HAR Dataset" 

In [ ]:
def load_feature_names(data_dir): 
    feats = pd.read_csv(data_dir / "features.txt", sep=r"s+", header=None, names=["idx", "name"]) 
    return [f"{name}__{idx}" for idx, name in zip(feats["idx"], feats["name"])] 

def load_activity_labels(data_dir): 
    acts = pd.read_csv(data_dir / "activity_labels.txt", sep=r"s+", header=None, names=["id", "activity"]) 
    return dict(zip(acts["id"], acts["activity"])) 

def load_split(data_dir, split, feature_names): 
    folder = data_dir / split 
    X = pd.read_csv(folder / f"X_{split}.txt", sep=r"s+", header=None, names=feature_names) 
    y = pd.read_csv(folder / f"y_{split}.txt", header=None).iloc[:, 0] 
    subject = pd.read_csv(folder / f"subject_{split}.txt", header=None).iloc[:, 0] 
    return X, y.rename("label"), subject.rename("subject") 

feature_names = load_feature_names(DATA_DIR) 
activity_map = load_activity_labels(DATA_DIR) 
class_names = [activity_map[i] for i in sorted(activity_map)] 
X_train, y_train, subj_train = load_split(DATA_DIR, "train", feature_names) 
X_test, y_test, subj_test = load_split(DATA_DIR, "test", feature_names)  

In [ ]:
scaler = StandardScaler() 
X_train_scaled = scaler.fit_transform(X_train) 
X_test_scaled = scaler.transform(X_test) 

In [ ]:
mlp = MLPClassifier( 
    hidden_layer_sizes=(128, 64), 
    activation="relu", 
    solver="adam", 
    max_iter=300, 
    random_state=RANDOM_STATE, 
    early_stopping=True, 
    n_iter_no_change=10 
) 

start = time.perf_counter() 
mlp.fit(X_train_scaled, y_train) 
fit_time = time.perf_counter() - start

In [ ]:
y_pred = mlp.predict(X_test_scaled) 
acc = accuracy_score(y_test, y_pred) 
macro_f1 = f1_score(y_test, y_pred, average="macro") 
print(f"Neural Network Test Accuracy: {acc:.4f}") 
print(f"Neural Network Test Macro F1: {macro_f1:.4f}") 
print(f"Training Time: {fit_time:.2f} seconds") 
print("nDetailed Classification Report:n") 
print(classification_report(y_test, y_pred, target_names=class_names)) 

In [ ]:
per_person = pd.Series(y_pred == y_test.values).groupby(subj_test.values).mean() 
print("nPer-Volunteer Accuracy on Unseen Test Subjects:") 
print(per_person.round(3))